### Do to high valeus in NANs, prefer to create dataset here :)

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import re
import warnings
warnings.filterwarnings('ignore')

class RemoteSensingMergeProcessor:
    """
    Pipeline for merging Spectral + GLCM + SAR data and comprehensive NaN preprocessing
    """
    
    def __init__(self):
        self.spectral_gdf = None
        self.glcm_gdf = None
        self.sar_gdf = None
        self.final_gdf = None
        self.nan_report = {}
        
    def load_and_merge_all_data(self, seg_shp, glcm_shp, sar_shp, band_names_file):
        """
        Load and merge all three datasets: Spectral + GLCM + SAR
        """
        print("🚀 STARTING DATA MERGING AND PREPROCESSING PIPELINE")
        print("=" * 80)
        
        # 1. Load Spectral Data
        print("\n1️⃣ LOADING SPECTRAL DATA")
        print("-" * 50)
        print(f"📂 Path: {seg_shp}")
        
        self.spectral_gdf = gpd.read_file(seg_shp)
        print(f"✅ Loaded successfully!")
        print(f"   📊 Segments: {len(self.spectral_gdf)}")
        print(f"   📋 Total columns: {len(self.spectral_gdf.columns)}")
        print(f"   🔢 Band columns: {len([c for c in self.spectral_gdf.columns if c.startswith('_b')])}")
        
        # 2. Map Spectral Band Names
        print(f"\n   🗺️ Mapping spectral band names...")
        self.spectral_gdf = self._map_spectral_names(self.spectral_gdf, band_names_file)
        
        # 3. Load GLCM Data
        print("\n2️⃣ LOADING GLCM DATA")
        print("-" * 50)
        print(f"📂 Path: {glcm_shp}")
        
        self.glcm_gdf = gpd.read_file(glcm_shp)
        print(f"✅ Loaded successfully!")
        print(f"   📊 Segments: {len(self.glcm_gdf)}")
        print(f"   📋 Total columns: {len(self.glcm_gdf.columns)}")
        print(f"   🔲 GLCM columns: {len([c for c in self.glcm_gdf.columns if c.startswith('_b')])}")
        
        # 4. Process GLCM Data
        print(f"\n   🔄 Processing GLCM data...")
        self.glcm_processed = self._process_glcm_data(band_names_file)
        
        # 5. Load SAR Data
        print("\n3️⃣ LOADING SAR DATA")
        print("-" * 50)
        print(f"📂 Path: {sar_shp}")
        
        self.sar_gdf = gpd.read_file(sar_shp)
        print(f"✅ Loaded successfully!")
        print(f"   📊 Segments: {len(self.sar_gdf)}")
        print(f"   📋 Total columns: {len(self.sar_gdf.columns)}")
        print(f"   📶 SAR columns: {len([c for c in self.sar_gdf.columns if c.startswith('_b')])}")
        
        # 6. Map SAR Band Names
        print(f"\n   🗺️ Mapping SAR band names...")
        self.sar_processed = self._map_sar_names(self.sar_gdf, band_names_file)
        
        # 7. Merge All Datasets
        print("\n4️⃣ MERGING ALL DATASETS")
        print("-" * 50)
        self._merge_all_datasets()
        
    def _map_spectral_names(self, seg_gdf, band_names_file):
        """
        Map generic band names to real spectral band names
        """
        with open(band_names_file, 'r') as f:
            all_band_names = [line.strip() for line in f.readlines()]
        
        optical_bands = [name for name in all_band_names if name.startswith('B')]
        print(f"      📡 Found {len(optical_bands)} optical bands in names file")
        
        # Create mapping
        band_name_mapping = {}
        band_cols = [col for col in seg_gdf.columns if col.startswith('_b')]
        
        for i, col in enumerate(band_cols):
            if i < len(optical_bands):
                band_name_mapping[col] = optical_bands[i]
        
        # Rename columns
        seg_gdf_renamed = seg_gdf.rename(columns=band_name_mapping)
        print(f"      ✅ Mapped {len(band_name_mapping)} spectral bands")
        print(f"      📋 Sample: {list(seg_gdf_renamed.columns)[1:6]}")
        
        return seg_gdf_renamed
    
    def _process_glcm_data(self, band_names_file):
        """
        Process GLCM data with proper naming and zero-to-NaN conversion
        """
        # Get spectral dates for mapping
        spectral_dates = []
        for col in self.spectral_gdf.columns:
            if col.startswith('B2_') and '_2021' in col:
                date_str = col.split('_')[1]
                if date_str not in spectral_dates:
                    spectral_dates.append(date_str)
        
        spectral_dates = sorted(spectral_dates)
        print(f"      📅 Found {len(spectral_dates)} unique dates for GLCM mapping")
        
        # GLCM metrics
        glcm_metrics = [
            'Mean', 'Variance', 'Homogeneity',
            'Contrast', 'Dissimilarity', 'Entropy',
            'Second_Moment', 'Correlation'
        ]
        
        # Build GLCM column mapping
        feature_mapping = {}
        glcm_cols = [col for col in self.glcm_gdf.columns if col.startswith('_b')]
        
        variance_variants = ['_varian', '_var', '_variance']
        
        for col in glcm_cols:
            try:
                band_num = int(col.split('_')[1][1:])
                date_idx = ((band_num - 1) // 8)
                metric_idx = (band_num - 1) % 8
                
                if date_idx < len(spectral_dates):
                    date = spectral_dates[date_idx]
                    metric = glcm_metrics[metric_idx]
                    
                    # Determine if it's mean or variance
                    is_variance = any(variant in col for variant in variance_variants)
                    stat_type = 'var' if is_variance else 'mean'
                    
                    feature_mapping[col] = f"GLCM_{date}_{metric}_{stat_type}"
            except:
                continue
        
        # Rename GLCM columns
        glcm_renamed = self.glcm_gdf[['PXLVAL'] + list(feature_mapping.keys())].copy()
        glcm_renamed = glcm_renamed.rename(columns=feature_mapping)
        
        # Convert zeros to NaN (GLCM convention)
        feature_cols = [col for col in glcm_renamed.columns if col != 'PXLVAL']
        total_zeros = 0
        
        for col in feature_cols:
            zero_count = (glcm_renamed[col] == 0).sum()
            total_zeros += zero_count
            glcm_renamed.loc[glcm_renamed[col] == 0, col] = np.nan
        
        print(f"      🔢 Mapped {len(feature_mapping)} GLCM features")
        print(f"      🔄 Converted {total_zeros:,} zeros to NaN")
        print(f"      📋 Sample: {[col for col in glcm_renamed.columns if 'GLCM' in col][:3]}")
        
        return glcm_renamed
    
    def _map_sar_names(self, sar_gdf, band_names_file):
        """
        Map generic SAR band names to real SAR band names
        """
        with open(band_names_file, 'r') as f:
            all_band_names = [line.strip() for line in f.readlines()]
        
        sar_bands = sorted([name for name in all_band_names if name.startswith(('VV_', 'VH_'))])
        print(f"      📶 Found {len(sar_bands)} SAR bands in names file")
        
        # Create SAR mapping
        sar_mapping = {}
        sar_cols = [col for col in sar_gdf.columns if col.startswith('_b')]
        
        for i, col in enumerate(sar_cols):
            if i < len(sar_bands):
                sar_mapping[col] = sar_bands[i]
        
        # Rename SAR columns
        sar_renamed = sar_gdf.rename(columns=sar_mapping)
        
        # Keep only PXLVAL and SAR features
        sar_features = [col for col in sar_renamed.columns if col.startswith(('VV_', 'VH_'))]
        sar_final = sar_renamed[['PXLVAL'] + sar_features]
        
        print(f"      ✅ Mapped {len(sar_mapping)} SAR bands")
        print(f"      📋 Sample: {sar_features[:225]}")
        
        return sar_final
    
    def _merge_all_datasets(self):
        """
        Merge Spectral + GLCM + SAR datasets
        """
        print("   🤝 Merging Spectral + GLCM + SAR datasets...")
        
        # Start with spectral data
        self.final_gdf = self.spectral_gdf.copy()
        print('Spec Done')
        # Merge GLCM
        self.final_gdf = self.final_gdf.merge(
            self.glcm_processed,
            on='PXLVAL',
            how='left'
        )
        print('GLCM Done')
        # Merge SAR
        self.final_gdf = self.final_gdf.merge(
            self.sar_processed,
            on='PXLVAL',
            how='left'
        )
        print('SAR Done')
        print(f"   ✅ Merge completed!")
        print(f"      📊 Final shape: {self.final_gdf.shape}")
        
        # Count feature types
        spectral_count = len([c for c in self.final_gdf.columns if re.match(r'B\d+_\d{8}', c)])
        glcm_count = len([c for c in self.final_gdf.columns if 'GLCM' in c])
        sar_count = len([c for c in self.final_gdf.columns if c.startswith(('VV_', 'VH_'))])
        
        print(f"      📡 Spectral bands: {spectral_count}")
        print(f"      🔲 GLCM features: {glcm_count}")
        print(f"      📶 SAR features: {sar_count}")
        print(f"      🔢 Total features: {spectral_count + glcm_count + sar_count}")
        
        # Sort columns chronologically
        self._sort_columns_chronologically()
        
        # Analyze NaN after merge
        self._analyze_nan_values("AFTER_MERGE")
    
    def _sort_columns_chronologically(self):
        """
        Sort columns by date for better organization
        """
        print("   📅 Sorting columns chronologically...")

        def get_date_from_column(col):
            # Extract date from various formats
            if '2021' in col:
                parts = col.split('_')
                for part in parts:
                    if part.startswith('2021') and len(part) == 8:
                        return part
            return '99999999'  # Non-date columns go to end
        
        # Separate columns by type
        id_cols = ['PXLVAL']
        geometry_cols = ['geometry'] if 'geometry' in self.final_gdf.columns else []
        date_cols = [col for col in self.final_gdf.columns if '2021' in col]
        other_cols = [col for col in self.final_gdf.columns 
                     if col not in id_cols + geometry_cols + date_cols]
        
        # Sort date columns
        sorted_date_cols = sorted(date_cols, key=get_date_from_column)
        
        # Reorder dataframe
        new_order = id_cols + sorted_date_cols + other_cols + geometry_cols
        self.final_gdf = self.final_gdf[new_order]
        
        print(f"      ✅ Columns sorted chronologically")
        print(f"      📅 Date range: {get_date_from_column(sorted_date_cols[0])} to {get_date_from_column(sorted_date_cols[-1])}")
    
    def _analyze_nan_values(self, stage):
        """
        Comprehensive NaN analysis
        """
        print(f"\n🔍 NaN ANALYSIS - {stage}")
        print("-" * 50)
        
        df = self.final_gdf
        total_cells = df.shape[0] * df.shape[1]
        nan_cells = df.isnull().sum().sum()
        nan_percentage = (nan_cells / total_cells) * 100
        
        print(f"   📊 Dataset shape: {df.shape}")
        print(f"   ❌ Total NaN cells: {nan_cells:,} ({nan_percentage:.2f}%)")
        
        # Analyze by feature type
        feature_types = {
            'Spectral': [c for c in df.columns if re.match(r'B\d+_\d{8}', c)],
            'GLCM': [c for c in df.columns if 'GLCM' in c],
            'SAR': [c for c in df.columns if c.startswith(('VV_', 'VH_'))]
        }
        
        for ftype, cols in feature_types.items():
            if cols:
                type_nan = df[cols].isnull().sum().sum()
                type_cells = len(df) * len(cols)
                type_percentage = (type_nan / type_cells) * 100
                print(f"   🔸 {ftype}: {type_nan:,}/{type_cells:,} NaN ({type_percentage:.2f}%)")
        
        # Find columns with high NaN percentage
        nan_cols = df.isnull().sum()
        high_nan_cols = nan_cols[nan_cols > len(df) * 0.5].sort_values(ascending=False)
        
        if len(high_nan_cols) > 0:
            print(f"   🚨 Columns with >50% NaN: {len(high_nan_cols)}")
            print("   🔝 Top 10 high-NaN columns:")
            for col, count in high_nan_cols.head(10).items():
                percentage = (count / len(df)) * 100
                print(f"      {col[:50]:50}: {count:4d} ({percentage:5.1f}%)")
        else:
            print("   ✅ No columns with >50% NaN values")
        
        self.nan_report[stage] = {
            'total_nan': nan_cells,
            'nan_percentage': nan_percentage,
            'high_nan_columns': len(high_nan_cols)
        }
    
    def preprocess_nan_values(self):
        """
        Comprehensive NaN preprocessing:
        1. Remove columns with >50% NaN
        2. Temporal interpolation (previous/next dates)
        3. Monthly mean filling
        4. Seasonal mean filling
        """
        print("\n5️⃣ COMPREHENSIVE NaN PREPROCESSING")
        print("-" * 50)
        
        # STEP 1: Remove columns with >50% NaN
        print("\n   📊 STEP 1: Removing columns with >50% NaN values")
        
        total_segments = len(self.final_gdf)
        threshold = total_segments * 0.5
        
        # Get all feature columns (exclude PXLVAL and geometry)
        feature_cols = [col for col in self.final_gdf.columns 
                       if col not in ['PXLVAL', 'geometry']]
        
        cols_to_drop = []
        cols_analysis = {'Spectral': 0, 'GLCM': 0, 'SAR': 0, 'Other': 0}
        
        for col in feature_cols:
            nan_count = self.final_gdf[col].isna().sum()
            
            if nan_count > threshold:
                cols_to_drop.append(col)
                
                # Categorize dropped column
                if re.match(r'B\d+_\d{8}', col):
                    cols_analysis['Spectral'] += 1
                    category = 'Spectral'
                elif 'GLCM' in col:
                    cols_analysis['GLCM'] += 1
                    category = 'GLCM'
                elif col.startswith(('VV_', 'VH_')):
                    cols_analysis['SAR'] += 1
                    category = 'SAR'
                else:
                    cols_analysis['Other'] += 1
                    category = 'Other'
                
                print(f"      🗑️ Dropping {category:8} column: {col[:50]:50} ({nan_count:4d}/{total_segments} NaN = {nan_count/total_segments*100:5.1f}%)")
        
        # Drop the columns
        self.final_gdf = self.final_gdf.drop(columns=cols_to_drop)
        
        print(f"\n   ✅ Dropped {len(cols_to_drop)} columns:")
        for ftype, count in cols_analysis.items():
            if count > 0:
                print(f"      🔸 {ftype}: {count} columns")
        
        remaining_features = [col for col in self.final_gdf.columns 
                             if col not in ['PXLVAL', 'geometry']]
        print(f"   📊 Remaining features: {len(remaining_features)}")
        
        # STEP 2: Temporal interpolation
        print(f"\n   🔄 STEP 2: Temporal interpolation (date-wise filling)")
        self._temporal_interpolation(remaining_features)
        
        # STEP 3: Monthly mean filling
        print(f"\n   📅 STEP 3: Monthly mean filling")
        self._monthly_mean_filling(remaining_features)
        
        # STEP 4: Seasonal mean filling
        print(f"\n   🍂 STEP 4: Seasonal mean filling")
        self._seasonal_mean_filling(remaining_features)
        
        # STEP 5: Final cleanup
        print(f"\n   🧹 STEP 5: Final cleanup with overall mean")
        self._final_nan_cleanup(remaining_features)
        
        # Final analysis
        self._analyze_nan_values("AFTER_PREPROCESSING")
    
    def _temporal_interpolation(self, feature_cols):
        """
        Fill NaN values using temporal interpolation (previous/next dates)
        """
        filled_count = 0
        
        for col in tqdm(feature_cols, desc="   Processing temporal interpolation"):
            if not self.final_gdf[col].isna().any():
                continue
            
            try:
                # Extract date from column name
                date_str = None
                if '2021' in col:
                    parts = col.split('_')
                    for part in parts:
                        if part.startswith('2021') and len(part) == 8:
                            date_str = part
                            break
                
                if not date_str:
                    continue
                
                # Find same-type columns from adjacent dates
                base_name = col.replace(date_str, '')
                
                # Find all dates for this feature type
                same_type_cols = [c for c in feature_cols if base_name in c and '2021' in c]
                same_type_dates = []
                
                for c in same_type_cols:
                    parts = c.split('_')
                    for part in parts:
                        if part.startswith('2021') and len(part) == 8:
                            same_type_dates.append((part, c))
                
                same_type_dates = sorted(same_type_dates)
                
                if len(same_type_dates) > 1:
                    # Find adjacent dates
                    current_idx = next((i for i, (d, c) in enumerate(same_type_dates) if c == col), -1)
                    
                    if current_idx >= 0:
                        adjacent_cols = []
                        if current_idx > 0:
                            adjacent_cols.append(same_type_dates[current_idx-1][1])
                        if current_idx < len(same_type_dates) - 1:
                            adjacent_cols.append(same_type_dates[current_idx+1][1])
                        
                        if adjacent_cols:
                            adjacent_mean = self.final_gdf[adjacent_cols].mean(axis=1)
                            before_fill = self.final_gdf[col].isna().sum()
                            self.final_gdf[col] = self.final_gdf[col].fillna(adjacent_mean)
                            after_fill = self.final_gdf[col].isna().sum()
                            filled_count += (before_fill - after_fill)
            
            except Exception as e:
                continue
        
        print(f"      ✅ Filled {filled_count:,} NaN values using temporal interpolation")
    
    def _monthly_mean_filling(self, feature_cols):
        """
        Fill NaN values using monthly mean
        """
        filled_count = 0
        
        for col in tqdm(feature_cols, desc="   Processing monthly mean filling"):
            if not self.final_gdf[col].isna().any():
                continue
            
            try:
                # Extract year-month from column
                date_str = None
                if '2021' in col:
                    parts = col.split('_')
                    for part in parts:
                        if part.startswith('2021') and len(part) == 8:
                            date_str = part
                            break
                
                if not date_str:
                    continue
                
                year_month = date_str[:6]  # YYYYMM
                base_name = col.replace(date_str, '')
                
                # Find all columns from same month and same feature type
                same_month_cols = []
                for c in feature_cols:
                    if base_name in c and year_month in c:
                        same_month_cols.append(c)
                
                if len(same_month_cols) > 1:
                    monthly_mean = self.final_gdf[same_month_cols].mean(axis=1)
                    before_fill = self.final_gdf[col].isna().sum()
                    self.final_gdf[col] = self.final_gdf[col].fillna(monthly_mean)
                    after_fill = self.final_gdf[col].isna().sum()
                    filled_count += (before_fill - after_fill)
            
            except Exception as e:
                continue
        
        print(f"      ✅ Filled {filled_count:,} NaN values using monthly mean")
    
    def _seasonal_mean_filling(self, feature_cols):
        """
        Fill NaN values using seasonal mean (current + adjacent months)
        """
        filled_count = 0
        
        for col in tqdm(feature_cols, desc="   Processing seasonal mean filling"):
            if not self.final_gdf[col].isna().any():
                continue
            
            try:
                # Extract year-month
                date_str = None
                if '2021' in col:
                    parts = col.split('_')
                    for part in parts:
                        if part.startswith('2021') and len(part) == 8:
                            date_str = part
                            break
                
                if not date_str:
                    continue
                
                year = int(date_str[:4])
                month = int(date_str[4:6])
                base_name = col.replace(date_str, '')
                
                # Calculate adjacent months
                prev_month = month - 1 if month > 1 else 12
                next_month = month + 1 if month < 12 else 1
                prev_year = year if month > 1 else year - 1
                next_year = year if month < 12 else year + 1
                
                # Create patterns for current and adjacent months
                patterns = [
                    f"{year}{month:02d}",
                    f"{prev_year}{prev_month:02d}",
                    f"{next_year}{next_month:02d}"
                ]
                
                # Find matching columns
                seasonal_cols = []
                for c in feature_cols:
                    if base_name in c:
                        for pattern in patterns:
                            if pattern in c:
                                seasonal_cols.append(c)
                                break
                
                if len(seasonal_cols) > 1:
                    seasonal_mean = self.final_gdf[seasonal_cols].mean(axis=1)
                    before_fill = self.final_gdf[col].isna().sum()
                    self.final_gdf[col] = self.final_gdf[col].fillna(seasonal_mean)
                    after_fill = self.final_gdf[col].isna().sum()
                    filled_count += (before_fill - after_fill)
            
            except Exception as e:
                continue
        
        print(f"      ✅ Filled {filled_count:,} NaN values using seasonal mean")
    
    def _final_nan_cleanup(self, feature_cols):
        """
        Final cleanup: fill remaining NaN with overall feature mean
        """
        filled_count = 0
        
        for col in feature_cols:
            if self.final_gdf[col].isna().any():
                before_fill = self.final_gdf[col].isna().sum()
                overall_mean = self.final_gdf[col].mean()
                self.final_gdf[col] = self.final_gdf[col].fillna(overall_mean)
                filled_count += before_fill
        
        print(f"      ✅ Filled {filled_count:,} remaining NaN values with overall mean")
    
    def generate_processing_report(self):
        """
        Generate comprehensive processing report
        """
        print("\n" + "=" * 80)
        print("📋 DATA PROCESSING REPORT")
        print("=" * 80)
        
        # Dataset overview
        print(f"\n📊 FINAL DATASET OVERVIEW:")
        print(f"   Total segments: {len(self.final_gdf)}")
        print(f"   Total features: {len(self.final_gdf.columns) - 2}")  # Exclude PXLVAL and geometry
        
        # Feature breakdown
        spectral_count = len([c for c in self.final_gdf.columns if re.match(r'B\d+_\d{8}', c)])
        glcm_count = len([c for c in self.final_gdf.columns if 'GLCM' in c])
        sar_count = len([c for c in self.final_gdf.columns if c.startswith(('VV_', 'VH_'))])
        
        print(f"\n📡 FEATURE BREAKDOWN:")
        print(f"   Spectral bands: {spectral_count}")
        print(f"   GLCM features: {glcm_count}")
        print(f"   SAR features: {sar_count}")
        print(f"   Total features: {spectral_count + glcm_count + sar_count}")
        
        # NaN processing summary
        print(f"\n❌ NaN PROCESSING SUMMARY:")
        for stage, stats in self.nan_report.items():
            print(f"   {stage}: {stats['nan_percentage']:.2f}% NaN values")
        
        # Final verification
        final_nan_count = self.final_gdf.isnull().sum().sum()
        print(f"\n✅ FINAL VERIFICATION:")
        print(f"   Remaining NaN values: {final_nan_count}")
        print(f"   Data quality: {'CLEAN' if final_nan_count == 0 else 'NEEDS ATTENTION'}")
        
        print(f"\n🎯 READY FOR NEXT STEPS:")
        print(f"   ✓ Calculate Vegetation Indices")
        print(f"   ✓ Assign labels")
        print(f"   ✓ Feature engineering")
        print(f"   ✓ Model training")
        
        print("=" * 80)


def main():
    """
    Main execution function for data merging and preprocessing
    """
    # Define paths
    raw_base = r"F:\_____My_Thesies____\_ImpFiles"
    
    seg_shp = os.path.join(raw_base, "3. Feature Selection", "_Sentinel Data", 
                          "2. Divided Data", "6.Segmentation", "Seg_2021", 
                          "1. Opt", "2021_Seg_With_Mean_Opt.shp")
    
    glcm_shp = os.path.join(raw_base, "3. Feature Selection", "_Sentinel Data",
                           "2. Divided Data", "6.Segmentation", "Seg_2021",
                           "4. GLCM", "2021_Seg_With_Mean_GLCM.shp")
    
    sar_shp = os.path.join(raw_base, "3. Feature Selection", "_Sentinel Data",
                          "2. Divided Data", "6.Segmentation", "Seg_2021",
                          "2. SAR", "2021_Seg_With_Mean_SAR.shp")
    
    band_names_file = os.path.join(raw_base, "3. Feature Selection", "_Sentinel Data",
                                  "2. Divided Data", "_Bands_Names", "2021 Bands Names.txt")
    
    # Initialize processor
    processor = RemoteSensingMergeProcessor()
    
    # Execute pipeline steps
    print("🎯 Starting comprehensive data merging and preprocessing...")
    
    # Step 1: Load and merge all data
    processor.load_and_merge_all_data(seg_shp, glcm_shp, sar_shp, band_names_file)
    
    # Step 2: Comprehensive NaN preprocessing
    processor.preprocess_nan_values()
    
    # Step 3: Generate final report
    processor.generate_processing_report()
    
    # Return the processed dataset for further use
    return processor.final_gdf

# Execute the pipeline
if __name__ == "__main__":
    print("🚀 EXECUTING REMOTE SENSING DATA PIPELINE")
    print("=" * 80)
    
    try:
        final_dataset = main()
        print(f"\n🎉 SUCCESS! Final dataset shape: {final_dataset.shape}")
        print(f"📊 Dataset ready for vegetation indices calculation and labeling")
        
        # Quick preview of the final dataset
        print(f"\n👀 QUICK PREVIEW:")
        print(f"First 5 columns: {list(final_dataset.columns)[:5]}")
        print(f"Last 5 columns: {list(final_dataset.columns)[-5:]}")
        
        # Sample data check
        feature_cols = [col for col in final_dataset.columns if col not in ['PXLVAL', 'geometry']]
        print(f"\n🔍 DATA QUALITY CHECK:")
        print(f"Total features: {len(feature_cols)}")
        print(f"Any remaining NaN: {final_dataset[feature_cols].isnull().sum().sum()}")
        print(f"Sample values from first feature:")
        print(final_dataset[feature_cols[0]].head())
        
    except Exception as e:
        print(f"❌ ERROR: {str(e)}")
        print("Please check file paths and data integrity")
        
    print("\n" + "=" * 80)
    print("🏁 PIPELINE EXECUTION COMPLETED")
    print("=" * 80)

🚀 EXECUTING REMOTE SENSING DATA PIPELINE
🎯 Starting comprehensive data merging and preprocessing...
🚀 STARTING DATA MERGING AND PREPROCESSING PIPELINE

1️⃣ LOADING SPECTRAL DATA
--------------------------------------------------
📂 Path: F:\_____My_Thesies____\_ImpFiles\3. Feature Selection\_Sentinel Data\2. Divided Data\6.Segmentation\Seg_2021\1. Opt\2021_Seg_With_Mean_Opt.shp
✅ Loaded successfully!
   📊 Segments: 6725
   📋 Total columns: 742
   🔢 Band columns: 740

   🗺️ Mapping spectral band names...
      📡 Found 740 optical bands in names file
      ✅ Mapped 740 spectral bands
      📋 Sample: ['B2_20210105', 'B3_20210105', 'B4_20210105', 'B5_20210105', 'B6_20210105']

2️⃣ LOADING GLCM DATA
--------------------------------------------------
📂 Path: F:\_____My_Thesies____\_ImpFiles\3. Feature Selection\_Sentinel Data\2. Divided Data\6.Segmentation\Seg_2021\4. GLCM\2021_Seg_With_Mean_GLCM.shp
✅ Loaded successfully!
   📊 Segments: 6725
   📋 Total columns: 1186
   🔲 GLCM columns: 1184



KeyboardInterrupt: 

In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import os
from typing import Dict, List, Tuple
import re
from tqdm import tqdm

class VegetationIndicesProcessor:
    """
    Process final dataset to calculate vegetation indices for each date
    """
    
    def __init__(self, final_dataset):
        """
        Initialize with the final merged dataset
        
        Args:
            final_dataset: GeoDataFrame containing merged spectral, GLCM, and SAR data
        """
        self.final_dataset = final_dataset.copy()
        self.spectral_columns = []
        self.date_groups = {}
        self.vi_columns = []
        
    def identify_spectral_bands(self):
        """
        Identify spectral band columns and group them by date
        """
        print("🔍 Identifying spectral bands and dates...")
        
        # Get all column names
        all_columns = self.final_dataset.columns.tolist()
        
        # Filter spectral columns (exclude GLCM and SAR)
        spectral_patterns = [
            r'.*B[2-9].*',  # Blue, Green, Red, NIR bands
            r'.*B1[1-2].*',  # SWIR bands
            r'.*B[5-8]A.*', # Red edge bands
        ]
        
        self.spectral_columns = []
        for col in all_columns:
            for pattern in spectral_patterns:
                if re.match(pattern, col, re.IGNORECASE):
                    # Exclude GLCM and SAR columns
                    if not any(x in col.upper() for x in ['GLCM', 'SAR', 'VV', 'VH']):
                        self.spectral_columns.append(col)
                        break
        
        print(f"📡 Found {len(self.spectral_columns)} spectral columns")
        
        # Extract dates from column names
        date_pattern = r'(\d{4})[-_]?(\d{2})[-_]?(\d{2})'
        
        for col in self.spectral_columns:
            date_match = re.search(date_pattern, col)
            if date_match:
                date_str = f"{date_match.group(1)}-{date_match.group(2)}-{date_match.group(3)}"
                
                if date_str not in self.date_groups:
                    self.date_groups[date_str] = []
                self.date_groups[date_str].append(col)
        
        print(f"📅 Found {len(self.date_groups)} unique dates:")
        for date, bands in self.date_groups.items():
            print(f"   {date}: {len(bands)} bands")
    
    def extract_band_order(self, band_columns: List[str]) -> List[str]:
        """
        Extract and order bands according to Sentinel-2 standard order
        Expected order: B02(Blue), B03(Green), B04(Red), B05(RedEdge1), 
                       B06(RedEdge2), B07(RedEdge3), B08(NIR), B8A(RedEdge4), 
                       B11(SWIR1), B12(SWIR2)
        """
        band_order = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
        ordered_bands = []
        
        for band_code in band_order:
            for col in band_columns:
                if band_code in col.upper():
                    ordered_bands.append(col)
                    break
        
        return ordered_bands
    
    def calculate_all_indices(self, image_data: np.ndarray) -> Dict[str, np.ndarray]:
        """
        Calculate 18 vegetation indices from Sentinel-2 bands
        
        Args:
            image_data: Array with shape (10, n_pixels) containing band values
        
        Returns:
            Dictionary of calculated indices
        """
        indices = {}
        
        # Convert to reflectance (assuming input is in DN or already scaled)
        blue = image_data[0] / 10000
        green = image_data[1] / 10000
        red = image_data[2] / 10000
        red_edge1 = image_data[3] / 10000
        red_edge2 = image_data[4] / 10000
        red_edge3 = image_data[5] / 10000
        nir = image_data[6] / 10000
        red_edge4 = image_data[7] / 10000
        swir1 = image_data[8] / 10000
        swir2 = image_data[9] / 10000
        
        L = 0.5  # Soil adjustment factor for SAVI
        
        # Calculate indices with error handling
        try:
            indices['NDVI'] = (nir - red) / (nir + red + 1e-6)
            indices['EVI'] = 2.5 * ((nir - red) / (nir + 6 * red - 7.5 * blue + 1))
            indices['SAVI'] = ((nir - red) / (nir + red + L)) * (1 + L)
            indices['TDVI'] = 1.5 * ((nir - red) / np.sqrt((nir ** 2 + red + 0.5)))
            indices['NDWI'] = (green - nir) / (green + nir + 1e-6)
            indices['2BDA'] = red_edge1 / (red + 1e-6)
            indices['NDCI'] = (red_edge1 - red) / (red_edge1 + red + 1e-6)
            indices['ENDVI'] = ((nir + green) - 2 * blue) / ((nir + green) + 2 * blue + 1e-6)
            indices['GNDVI'] = (nir - green) / (nir + green + 1e-6)
            indices['NDRE'] = (nir - red_edge2) / (nir + red_edge2 + 1e-6)
            indices['GRVI'] = nir / (green + 1e-6)
            indices['MSAVI'] = (2 * nir + 1 - np.sqrt((2 * nir + 1) ** 2 - 8 * (nir - red))) / 2
            indices['LSWI'] = (nir - swir1) / (nir + swir1 + 1e-6)
            indices['NDMI'] = (nir - swir1) / (nir + swir1 + 1e-6)
            indices['MNDWI'] = (green - swir1) / (green + swir1 + 1e-6)
            
            # Calculate MBI first for EMBI calculation
            indices['MBI'] = ((swir1 - swir2 - nir) / (swir1 + swir2 + nir + 1e-6)) + 0.5
            indices['EMBI'] = (indices['MBI'] - indices['MNDWI'] - 0.5) / (indices['MBI'] + indices['MNDWI'] + 1.5)
            
            # MCARI2 with more robust calculation
            sqrt_term = np.sqrt(np.maximum((2 * nir + 1) ** 2 - (6 * nir - 5 * np.sqrt(np.maximum(red, 0) + 1e-6)), 1e-6))
            indices['MCARI2'] = 1.5 * (2.5 * (nir - red) - 1.3 * (nir - green)) / (sqrt_term - 0.5 + 1e-6)
            
        except Exception as e:
            print(f"⚠️ Warning in index calculation: {e}")
            # Fill with NaN if calculation fails
            for key in ['NDVI', 'EVI', 'SAVI', 'TDVI', 'NDWI', '2BDA', 'NDCI', 'ENDVI', 
                       'GNDVI', 'NDRE', 'GRVI', 'MSAVI', 'LSWI', 'NDMI', 'MNDWI', 'MBI', 'EMBI', 'MCARI2']:
                if key not in indices:
                    indices[key] = np.full_like(nir, np.nan)
        
        return indices
    
    def process_vegetation_indices(self):
        """
        Calculate vegetation indices for each date and add to dataset
        """
        print("🌱 Calculating vegetation indices for each date...")
        
        final_dataset_with_vis = self.final_dataset.copy()
        
        for date, band_columns in tqdm(self.date_groups.items(), desc="Processing dates"):
            print(f"\n📅 Processing date: {date}")
            
            # Order bands correctly
            ordered_bands = self.extract_band_order(band_columns)
            
            if len(ordered_bands) < 10:
                print(f"⚠️ Warning: Only {len(ordered_bands)} bands found for {date}, skipping...")
                continue
            
            print(f"   Using bands: {[col.split('_')[-1] if '_' in col else col for col in ordered_bands[:5]]}...")
            
            # Extract band data for all segments
            try:
                band_data = []
                for band_col in ordered_bands[:10]:  # Use first 10 ordered bands
                    if band_col in self.final_dataset.columns:
                        band_data.append(self.final_dataset[band_col].values)
                    else:
                        print(f"⚠️ Warning: Band {band_col} not found")
                        band_data.append(np.zeros(len(self.final_dataset)))
                
                # Convert to numpy array: shape (n_bands, n_segments)
                band_array = np.array(band_data)
                
                # Calculate indices for all segments at once
                indices_dict = self.calculate_all_indices(band_array)
                
                # Add indices to dataset with date prefix
                for idx_name, idx_values in indices_dict.items():
                    col_name = f"{idx_name}_{date}"
                    final_dataset_with_vis[col_name] = idx_values
                    self.vi_columns.append(col_name)
                    
            except Exception as e:
                print(f"❌ Error processing date {date}: {e}")
                continue
        
        self.final_dataset_with_vis = final_dataset_with_vis
        print(f"\n✅ Added {len(self.vi_columns)} vegetation index columns")
    
    def generate_summary_report(self):
        """
        Generate comprehensive summary of the enhanced dataset
        """
        print("\n" + "="*80)
        print("📋 ENHANCED DATASET SUMMARY REPORT")
        print("="*80)
        
        total_features = len(self.final_dataset_with_vis.columns) - 1  # Exclude geometry
        
        print(f"📊 FINAL ENHANCED DATASET:")
        print(f"   Total segments: {len(self.final_dataset_with_vis)}")
        print(f"   Total features: {total_features}")
        
        print(f"\n📡 FEATURE BREAKDOWN:")
        print(f"   Spectral bands: {len(self.spectral_columns)}")
        print(f"   Vegetation indices: {len(self.vi_columns)}")
        
        # Count other features
        glcm_features = len([col for col in self.final_dataset_with_vis.columns if 'GLCM' in col.upper()])
        sar_features = len([col for col in self.final_dataset_with_vis.columns if any(x in col.upper() for x in ['SAR', 'VV', 'VH'])])
        
        print(f"   GLCM features: {glcm_features}")
        print(f"   SAR features: {sar_features}")
        
        print(f"\n🌱 VEGETATION INDICES BY DATE:")
        dates_with_vis = list(set([col.split('_')[0] for col in self.vi_columns]))
        for date in sorted(dates_with_vis):
            date_vis = [col for col in self.vi_columns if col.startswith(date)]
            print(f"   {date}: {len(date_vis)} indices")
        
        print(f"\n✅ DATA QUALITY:")
        nan_count = self.final_dataset_with_vis.isnull().sum().sum()
        print(f"   Total NaN values: {nan_count}")
        print(f"   Data completeness: {((total_features * len(self.final_dataset_with_vis) - nan_count) / (total_features * len(self.final_dataset_with_vis)) * 100):.2f}%")
    

# Main execution function
def enhance_dataset_with_vegetation_indices(final_dataset, output_base_path):
    """
    Main function to enhance dataset with vegetation indices
    
    Args:
        final_dataset: The merged GeoDataFrame from previous processing
        output_base_path: Base path for saving enhanced dataset
    
    Returns:
        Enhanced GeoDataFrame with vegetation indices
    """
    
    print("🚀 Starting vegetation indices enhancement...")
    
    # Initialize processor
    processor = VegetationIndicesProcessor(final_dataset)
    
    # Step 1: Identify spectral bands and dates
    processor.identify_spectral_bands()
    
    # Step 2: Calculate vegetation indices
    processor.process_vegetation_indices()
    
    # Step 3: Generate summary report
    processor.generate_summary_report()
    
    print(f"\n🎉 Enhancement complete!")
    
    return processor.final_dataset_with_vis

# Execute the enhancement
if __name__ == "__main__":
    # Assuming final_dataset is available from previous processing
    output_path = r"F:\_____My_Thesies____\Implimantation\3.Train_Model\_Base_Data\Predict\2021"
    
    # Enhance dataset with vegetation indices
    final_dataset_with_VIs = enhance_dataset_with_vegetation_indices(final_dataset, output_path)
    
    print("\n✅ Dataset enhancement completed successfully!")
    print(f"Enhanced dataset shape: {final_dataset_with_VIs.shape}")

🚀 Starting vegetation indices enhancement...
🔍 Identifying spectral bands and dates...
📡 Found 600 spectral columns
📅 Found 60 unique dates:
   2020-01-06: 10 bands
   2020-01-16: 10 bands
   2020-01-21: 10 bands
   2020-01-26: 10 bands
   2020-01-31: 10 bands
   2020-02-05: 10 bands
   2020-02-10: 10 bands
   2020-03-01: 10 bands
   2020-03-06: 10 bands
   2020-03-11: 10 bands
   2020-03-31: 10 bands
   2020-04-05: 10 bands
   2020-04-10: 10 bands
   2020-04-15: 10 bands
   2020-04-20: 10 bands
   2020-04-30: 10 bands
   2020-05-05: 10 bands
   2020-05-10: 10 bands
   2020-05-15: 10 bands
   2020-05-20: 10 bands
   2020-05-25: 10 bands
   2020-05-30: 10 bands
   2020-06-04: 10 bands
   2020-06-09: 10 bands
   2020-06-14: 10 bands
   2020-06-19: 10 bands
   2020-06-24: 10 bands
   2020-06-29: 10 bands
   2020-07-04: 10 bands
   2020-07-09: 10 bands
   2020-07-14: 10 bands
   2020-07-19: 10 bands
   2020-07-24: 10 bands
   2020-07-29: 10 bands
   2020-08-03: 10 bands
   2020-08-08: 10 b

Processing dates:  15%|█▌        | 9/60 [00:00<00:00, 81.66it/s]


📅 Processing date: 2020-01-06
   Using bands: ['20200106', '20200106', '20200106', '20200106', '20200106']...

📅 Processing date: 2020-01-16
   Using bands: ['20200116', '20200116', '20200116', '20200116', '20200116']...

📅 Processing date: 2020-01-21
   Using bands: ['20200121', '20200121', '20200121', '20200121', '20200121']...

📅 Processing date: 2020-01-26
   Using bands: ['20200126', '20200126', '20200126', '20200126', '20200126']...

📅 Processing date: 2020-01-31
   Using bands: ['20200131', '20200131', '20200131', '20200131', '20200131']...

📅 Processing date: 2020-02-05
   Using bands: ['20200205', '20200205', '20200205', '20200205', '20200205']...

📅 Processing date: 2020-02-10
   Using bands: ['20200210', '20200210', '20200210', '20200210', '20200210']...

📅 Processing date: 2020-03-01
   Using bands: ['20200301', '20200301', '20200301', '20200301', '20200301']...

📅 Processing date: 2020-03-06
   Using bands: ['20200306', '20200306', '20200306', '20200306', '20200306']...



Processing dates:  30%|███       | 18/60 [00:00<00:00, 78.78it/s]


📅 Processing date: 2020-05-10
   Using bands: ['20200510', '20200510', '20200510', '20200510', '20200510']...

📅 Processing date: 2020-05-15
   Using bands: ['20200515', '20200515', '20200515', '20200515', '20200515']...

📅 Processing date: 2020-05-20
   Using bands: ['20200520', '20200520', '20200520', '20200520', '20200520']...

📅 Processing date: 2020-05-25
   Using bands: ['20200525', '20200525', '20200525', '20200525', '20200525']...

📅 Processing date: 2020-05-30
   Using bands: ['20200530', '20200530', '20200530', '20200530', '20200530']...

📅 Processing date: 2020-06-04
   Using bands: ['20200604', '20200604', '20200604', '20200604', '20200604']...

📅 Processing date: 2020-06-09
   Using bands: ['20200609', '20200609', '20200609', '20200609', '20200609']...

📅 Processing date: 2020-06-14
   Using bands: ['20200614', '20200614', '20200614', '20200614', '20200614']...

📅 Processing date: 2020-06-19
   Using bands: ['20200619', '20200619', '20200619', '20200619', '20200619']...


...

Processing dates:  85%|████████▌ | 51/60 [00:00<00:00, 46.17it/s]


📅 Processing date: 2020-10-12
   Using bands: ['20201012', '20201012', '20201012', '20201012', '20201012']...

📅 Processing date: 2020-10-17
   Using bands: ['20201017', '20201017', '20201017', '20201017', '20201017']...

📅 Processing date: 2020-10-22
   Using bands: ['20201022', '20201022', '20201022', '20201022', '20201022']...

📅 Processing date: 2020-10-27
   Using bands: ['20201027', '20201027', '20201027', '20201027', '20201027']...

📅 Processing date: 2020-11-01
   Using bands: ['20201101', '20201101', '20201101', '20201101', '20201101']...

📅 Processing date: 2020-11-06
   Using bands: ['20201106', '20201106', '20201106', '20201106', '20201106']...

📅 Processing date: 2020-11-16
   Using bands: ['20201116', '20201116', '20201116', '20201116', '20201116']...

📅 Processing date: 2020-11-26
   Using bands: ['20201126', '20201126', '20201126', '20201126', '20201126']...

📅 Processing date: 2020-12-01
   Using bands: ['20201201', '20201201', '20201201', '20201201', '20201201']...


Processing dates: 100%|██████████| 60/60 [00:01<00:00, 49.92it/s]


📅 Processing date: 2020-12-11
   Using bands: ['20201211', '20201211', '20201211', '20201211', '20201211']...

📅 Processing date: 2020-12-21
   Using bands: ['20201221', '20201221', '20201221', '20201221', '20201221']...

📅 Processing date: 2020-12-26
   Using bands: ['20201226', '20201226', '20201226', '20201226', '20201226']...

📅 Processing date: 2020-12-31
   Using bands: ['20201231', '20201231', '20201231', '20201231', '20201231']...

✅ Added 1080 vegetation index columns

📋 ENHANCED DATASET SUMMARY REPORT
📊 FINAL ENHANCED DATASET:
   Total segments: 6586
   Total features: 2917

📡 FEATURE BREAKDOWN:
   Spectral bands: 600
   Vegetation indices: 1080
   GLCM features: 998
   SAR features: 238

🌱 VEGETATION INDICES BY DATE:
   2BDA: 60 indices
   EMBI: 60 indices
   ENDVI: 60 indices
   EVI: 60 indices
   GNDVI: 60 indices
   GRVI: 60 indices
   LSWI: 60 indices
   MBI: 60 indices
   MCARI2: 60 indices
   MNDWI: 60 indices
   MSAVI: 60 indices
   NDCI: 60 indices
   NDMI: 60 indic

   Total NaN values: 0
   Data completeness: 100.00%

🎉 Enhancement complete!

✅ Dataset enhancement completed successfully!
Enhanced dataset shape: (6586, 2918)


In [3]:
output_dir = r"F:\_____My_Thesies____\Implimantation\3.Train_Model\_Base_Data\Predict\2021"

csv_path = os.path.join(output_dir, "final_dataset_with_VIs.csv")
final_dataset_with_VIs.to_csv(csv_path, index=False)
print(f"✅ CSV with geometry saved to: {csv_path}")

✅ CSV with geometry saved to: F:\_____My_Thesies____\Implimantation\3.Train_Model\_Base_Data\Predict\2020\final_dataset_with_VIs.csv
